Running this notebook requires that a kafka server is running on the local system:

```
docker kill kafka; docker rm kafka;  docker run -d --name kafka -p 9092:9092 apache/kafka:3.7.0
```
The first two command (kill and rm) are just to clean up any existing kafka container, and the third command starts a new kafka server in a docker container.

Then run the `producer.py` script in this same directory to produce randomly sized batches of 
messages with random delays in between batches.

Once both are running, you can run this notebook. 

Re-running the producer.py script while this notebook is running will cause new batches to 
be produced and consumed by the notebook, demonstrating that the notebook can handle batches
of varying sizes and delays and will wait for long periods of time between batches.

Restarting this notebook doesn't seem to pick up new messages off the stream though.
It seems to get stuck "Waiting for samples".

In [ ]:
import json
import time
import numpy as np
from confluent_kafka import Consumer, KafkaError
from hyrax import Hyrax

BATCH_SIZE = 10
FLUSH_TIMEOUT_SECS = 5.0


def kafka_batches(consumer, batch_size, flush_timeout=FLUSH_TIMEOUT_SECS):
    """Yield dicts in the infer_stream batch format, accumulated from Kafka messages."""
    batch_ids, batch_images = [], []
    last_message_time = time.monotonic()
    while True:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            # Flush a partial batch after a quiet period
            if batch_ids and (time.monotonic() - last_message_time) >= flush_timeout:
                print(f"No messages for {flush_timeout}s — flushing {len(batch_ids)} item(s)")
                yield {"object_id": batch_ids, "data": {"image": np.stack(batch_images, dtype=np.float32)}}
                batch_ids, batch_images = [], []
                last_message_time = time.monotonic()
            continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF:
                print(f"[consumer] Error: {msg.error()}")
            continue

        payload = json.loads(msg.value().decode())
        batch_ids.append(msg.key().decode())
        print(f"Received message with key {batch_ids[-1]} and payload {payload}")
        batch_images.append(np.array(payload["value"], dtype=np.float32))  # adjust to your message schema

        if len(batch_ids) >= batch_size:
            yield {
                "object_id": np.asarray(batch_ids, dtype=np.str_),
                "data": {"image": np.stack(batch_images, dtype=np.float32)},
            }
            batch_ids, batch_images = [], []
            last_message_time = time.monotonic()


consumer = Consumer(
    {
        "bootstrap.servers": "localhost:9092",
        "group.id": "infer-stream-group",
        "auto.offset.reset": "earliest",
    }
)
consumer.subscribe(["my-topic"])

# A representative batch to initialize model architecture
print("Waiting for samples")
sample = next(kafka_batches(consumer, BATCH_SIZE))

print("Starting Hyrax")
hy = Hyrax()
print("Setting configs")
hy.set_config("model.name", "HyraxLoopback")
hy.set_config("infer_stream.model_weights_file", "foo.bar")
print("Setting up context manager")
with hy.infer_stream(sample_batch=sample) as session:
    # Process the sample we already pulled, then keep going
    session.process(sample)
    for batch in kafka_batches(consumer, BATCH_SIZE):
        results = session.process(batch)
        print(f"processed {len(batch['object_id'])} items, output shape {results.shape}")

consumer.close()